# Preprocess TRE Diet Logs

This notebook converts raw participant diet events into participant/time-window feature tables.

Supported feature modes:

- `enriched`: amount-scaled per-100 g enriched food features.
- `embedding`: amount-weighted food-card embedding vectors.
- `kg`: amount-weighted KG/downstream feature exports.

Only run this inside TRE when participant diet logs are involved.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from downstream_analysis.preprocess import build_design_matrix, read_table

PROJECT_ROOT

## User Settings

Set `DIET_EVENTS_PATH` to the raw TRE diet-event file. The legacy diet logging table usually contains participant id, `food_id`, `weight_g`, and a collection/local timestamp.

In [ ]:
# Required TRE input
DIET_EVENTS_PATH = PROJECT_ROOT / 'tre_inputs' / 'diet_events.csv'

# Choose one: enriched, embedding, kg
FEATURE_MODE = 'embedding'

# Common columns in the TRE diet-event file
ID_COL = 'participant_id'
FOOD_COL = 'food_id'
GRAMS_COL = 'weight_g'
TIME_COL = 'collection_timestamp'  # local_timestamp or collection_date also work

# Choose one: participant, day, week, month, year
WINDOW = 'participant'

# Reference tables generated outside TRE
ENRICHED_REFERENCE_PATH = PROJECT_ROOT / 'outputs' / 'enhanced_hpp' / '1.denovo' / 'hpp_feature_matrix_per_100g.csv'
EMBEDDING_REFERENCE_PATH = PROJECT_ROOT / 'outputs' / 'food_card' / 'denovo' / 'embeddings' / 'hpp_food_card_embeddings_full_biology_text_text_embedding_3_large.parquet'
KG_REFERENCE_PATH = PROJECT_ROOT / 'outputs' / 'downstream_features' / 'denovo' / 'broad_diet_health' / 'hpp_downstream_feature_table.csv'

OUTPUT_DIR = PROJECT_ROOT / 'downstream_analysis' / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

reference_by_mode = {
    'enriched': ENRICHED_REFERENCE_PATH,
    'embedding': EMBEDDING_REFERENCE_PATH,
    'kg': KG_REFERENCE_PATH,
}
FOOD_REFERENCE_PATH = reference_by_mode[FEATURE_MODE]
OUTPUT_PATH = OUTPUT_DIR / f'X_{FEATURE_MODE}_{WINDOW}.parquet'

print('Diet events:', DIET_EVENTS_PATH)
print('Food reference:', FOOD_REFERENCE_PATH)
print('Output:', OUTPUT_PATH)

## Build Design Matrix

In [ ]:
summary = build_design_matrix(
    diet_events_path=DIET_EVENTS_PATH,
    food_reference_path=FOOD_REFERENCE_PATH,
    output_path=OUTPUT_PATH,
    feature_mode=FEATURE_MODE,
    id_col=ID_COL,
    food_col=FOOD_COL,
    grams_col=GRAMS_COL,
    time_col=TIME_COL,
    window=WINDOW,
)
summary

## Inspect Output

In [ ]:
X = read_table(OUTPUT_PATH)
print(X.shape)
X.head()